# Gemma 4 12B QAT Multimodal Chat

Run the Gemma 4 12B QAT GGUF model locally in Google Colab with the pre-built CUDA wheel for `llama-cpp-python`.

Use a GPU runtime before running this notebook: **Runtime > Change runtime type > T4 GPU**.

Current Colab CUDA images commonly provide CUDA 12 user-space libraries even when `nvidia-smi` reports a CUDA 13-capable driver, so this notebook installs the `cu125` wheel. If your runtime provides `libcudart.so.13`, switch the wheel index URL to `/whl/cu130`.


In [ ]:
!pip install --no-cache-dir --upgrade --force-reinstall \
  "huggingface-hub>=0.23.0" \
  llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu125


In [ ]:
from llama_cpp import Llama
from llama_cpp.llama_chat_format import Gemma4ChatHandler

MODEL_REPO = "unsloth/gemma-4-12B-it-qat-GGUF"
MODEL_FILE = "gemma-4-12B-it-qat-UD-Q4_K_XL.gguf"
MMPROJ_FILE = "mmproj-F16.gguf"

chat_handler = Gemma4ChatHandler.from_pretrained(
    repo_id=MODEL_REPO,
    filename=MMPROJ_FILE,
    verbose=False,
)

llm = Llama.from_pretrained(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    chat_handler=chat_handler,
    n_gpu_layers=-1,
    n_ctx=8192,
    flash_attn=True,
    verbose=False,
)


In [ ]:
from IPython.display import Image, display

IMAGE_URL = "https://raw.githubusercontent.com/ggml-org/llama.cpp/master/tools/mtmd/test-1.jpeg"

display(Image(url=IMAGE_URL, width=320))


In [ ]:
response = llm.create_chat_completion(
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe this image in one concise sentence."},
                {"type": "image_url", "image_url": {"url": IMAGE_URL}},
            ],
        }
    ],
    max_tokens=128,
    temperature=0.2,
)

print(response["choices"][0]["message"]["content"])


In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "record_image_observation",
            "description": "Record a structured observation about an image.",
            "parameters": {
                "type": "object",
                "properties": {
                    "main_subject": {"type": "string"},
                    "setting": {"type": "string"},
                    "notable_details": {
                        "type": "array",
                        "items": {"type": "string"},
                    },
                    "confidence": {"type": "number", "minimum": 0, "maximum": 1},
                },
                "required": ["main_subject", "setting", "notable_details", "confidence"],
            },
        },
    }
]

response = llm.create_chat_completion(
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Use the provided tool to record a structured observation for this image."},
                {"type": "image_url", "image_url": {"url": IMAGE_URL}},
            ],
        }
    ],
    tools=tools,
    tool_choice={"type": "function", "function": {"name": "record_image_observation"}},
    max_tokens=256,
    temperature=0.0,
)

message = response["choices"][0]["message"]
print(message.get("tool_calls", message.get("content")))
